In [ ]:
import numpy as np
import pandas as pd
#matrixial form
# 1 matriz por layer, logo
class DenseNeuralNetwork():
    def __init__(self, layer_sizes, learning_rate=0.01):
        self.weights = []
        self.biases = [] # Added biases for better learning capability
        self.learning_rate = learning_rate
        
        for i in range(len(layer_sizes) - 1):
            # Using standard normal initialization
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) 
            b = np.random.randn(layer_sizes[i+1], 1)
            self.weights.append(w)
            self.biases.append(b)

    def _relu(self,x):
        return max(0,x)
    
    def _fowardpass(self,X:np.array,n):
        return np.dot(self.weights[n],X)
    
    def _sigmoid(self,X):
        return 1/(1+np.exp(-X))
    
    def _sigmoid_derivative(self,a):
        return a*(1-a)
    
    def foward(self,x):
        # Reshape input to be a column vector
        a = x.reshape(-1, 1)
        activations = [a] # Store input and all layer activations
        
        for w, b in zip(self.weights, self.biases):
            z = np.dot(w, a) + b
            a = self._sigmoid(z)
            activations.append(a)
            
        return activations
    
    
    def _EQM(self,x,y):
        return (x-y)**2
    
    def _EQM_derivado(self,x,y):
        return 2*(x-y)

    def backward(self, activations, target):
        target = target.reshape(-1, 1)
        
        # Calculate error of the output layer
        output_activation = activations[-1]
        
        # Element-wise product of cost derivative and activation derivative
        delta = self._EQM_derivado(output_activation, target) * self._sigmoid_derivative(output_activation)
        
        # Gradients for the output layer
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        
        nabla_w[-1] = np.dot(delta, activations[-2].T)
        nabla_b[-1] = delta
        
        # Backpropagate the error to previous layers
        for l in range(2, len(self.weights) + 1):
            # delta mapped backwards through weights
            delta = np.dot(self.weights[-l+1].T, delta) * self._sigmoid_derivative(activations[-l])
            nabla_w[-l] = np.dot(delta, activations[-l-1].T)
            nabla_b[-l] = delta
            
        return nabla_w, nabla_b

    def update_parameters(self, nabla_w, nabla_b):
        # Update weights and biases using gradient descent
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * nabla_w[i]
            self.biases[i] -= self.learning_rate * nabla_b[i]

    def train(self, X, Y, epochs=1000):
        for epoch in range(epochs):
            total_loss = 0
            for i in range(len(X)):
                # Forward pass
                activations = self.foward(X[i])
                
                # Compute Loss
                prediction = activations[-1]
                target = Y[i]
                total_loss += np.mean((prediction.flatten() - target)**2)
                
                # Backward pass
                nabla_w, nabla_b = self.backward(activations, target)
                
                # Update weights
                self.update_parameters(nabla_w, nabla_b)
                
            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {total_loss/len(X):.4f}")

        

        
        


In [42]:
NN = DenseNeuralNetwork((2,30,40,1))


In [43]:
NN.foward(np.array([3,2]))[-1]

array([[0.00799866]])

In [44]:
data = {"input":[],
        "output":[]}

R = 49

for i in range(10):
    for j in range(10):
        data['input'].append(np.array([i,j]))
        if i**2 + j**2 > R**2:
            data['output'].append(0)
        else:
            data['output'].append(1)

data = pd.DataFrame(data)


In [45]:
NN.train(data['input'],data['output'])

Epoch 0, Loss: 0.9799
Epoch 100, Loss: 0.0001
Epoch 200, Loss: 0.0001
Epoch 300, Loss: 0.0000
Epoch 400, Loss: 0.0000
Epoch 500, Loss: 0.0000
Epoch 600, Loss: 0.0000
Epoch 700, Loss: 0.0000
Epoch 800, Loss: 0.0000
Epoch 900, Loss: 0.0000


In [46]:
NN.foward(np.array([3,2]))[-1]

array([[0.99800029]])